# MANESKILL — statistical analysis

**Active versus passive learning for non-technical skills acquisition in medical students under stress: the MANESKILL randomized simulation study** (JMIR Medical Education, manuscript #97016).

Sessions (teams of 3–9 students) were the unit of randomisation and of intervention; the Anaesthetists' Non-Technical Skills (ANTS) score was rated per participant. This notebook reproduces every number, table and figure of the manuscript:

| Section | Manuscript item |
|---|---|
| 1. Data preparation | de-identified analysis dataset (`data/analysis.csv`) |
| 2. Cluster structure | ICC, design effect, effective sample size — Results, **Supplementary Table 3** |
| 3. Mixed models (R: `lmm_cluster.R`) | primary and secondary outcomes — **Table 2**, Results text |
| 4. Session-level analyses | sensitivity analysis and task outcomes — **Table 3**, Results text |
| 5. Sensitivity: allocation stage | **Supplementary Table 6** |
| 6. Descriptive tables | **Table 1**, **Supplementary Tables 4–5**, individual-level counts |
| 7. Figures | **Figure 1**, **Figure 2**, **Supplementary Figure 1** (CONSORT), **Supplementary Figure 4** |
| 8. Verification | asserts the key numbers reported in the manuscript |

**Requirements.** Python ≥ 3.11 with `pandas`, `numpy`, `scipy`, `matplotlib`, `openpyxl` (see `requirements.txt`) and R ≥ 4.1 with `lme4`, `lmerTest`, `pbkrtest`, `emmeans` available as `Rscript` on the PATH.

**Data.** The raw export `data/df.xlsx` is not distributed (it contains dates of birth, rotation sites and free-text comments; available from the corresponding author under a data-sharing agreement). Section 1 builds the de-identified `data/analysis.csv` from it when it is present; otherwise the notebook starts from `data/analysis.csv` if it has been provided.

In [1]:
import os, subprocess, shutil, sys, platform
import numpy as np, pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
ARMS = ["Control", "Passive", "Active"]
PAIRS = [("Passive", "Control"), ("Active", "Control"), ("Active", "Passive")]
LABELS = {"Control": "Control", "Passive": "Passive\nlearning", "Active": "Active\nlearning"}
COLORS = {"Control": "#b8b8b8", "Passive": "#8fc1e3", "Active": "#2b6ca3"}
DATA, OUT = "data", "outputs"
os.makedirs(OUT, exist_ok=True)
RAW = f"{DATA}/df.xlsx"            # raw export (not distributed)
ANALYSIS = f"{DATA}/analysis.csv"  # de-identified participant-level dataset built in section 1
SEED = 2026
rng = np.random.default_rng(SEED)  # permutation tests and jitter in figures

# Participant flow figures that are not in the dataset (CONSORT diagram, Supplementary Figure 1)
N_APPROACHED = 102          # students who presented for a session
N_NOT_ENROLLED = 2          # presented for the last session after the target sample size was reached
EXCLUDED = {"Control": 1}   # 1 control student left before the end of the procedure (not in the dataset)

# Figure 1 brackets: "change" = mixed models of the change score (as stated in the figure legend);
# "ancova" = phase-2 score adjusted for phase 1 (the contrasts reported in Table 2)
FIG1_CONTRASTS = "change"

rscript = shutil.which("Rscript")
assert rscript, "Rscript not found: install R with lme4, lmerTest, pbkrtest and emmeans"
print("Python", platform.python_version(), "| pandas", pd.__version__, "| scipy", __import__("scipy").__version__, "| matplotlib", matplotlib.__version__)
print(subprocess.run([rscript, "-e", 'cat(R.version.string, "| lme4", as.character(packageVersion("lme4")), "| emmeans", as.character(packageVersion("emmeans")))'], capture_output=True, text=True).stdout)

Python 3.11.15 | pandas 3.0.2 | scipy 1.17.1 | matplotlib 3.10.9


R version 4.3.3 (2024-02-29) | lme4 1.1.35.1 | emmeans 1.10.0


## 1. Data preparation

From the raw export, one row per analysed participant. ANTS: 7 behavioural markers scored 0–4; each domain score is the mean of its markers (situation awareness 2, decision making 2, team working 2, task management 1) and the total is the sum of the four domains (0–16). Sessions are recoded `S01`–`S16` in chronological order and the session date to a date index `D01`–`D09`, so that the analysis dataset carries no direct identifier.

In [2]:
def build_analysis(raw_path):
    x = pd.read_excel(raw_path)
    x = x[x["randomisation"].notna()].copy()
    x["arm"] = x["randomisation"].astype(str).str.strip().map({"Rien": "Control", "Cours": "Passive", "Parole": "Active"})
    x["session_id"] = x["#groupe"].astype(str)
    x["session_date"] = pd.to_datetime(x["date_jeu"])
    # ANTS domains and total
    x["cal_sa"] = x[["cal_ants_cs_changement_situatoin", "cal_ants_cs_anticipe"]].mean(axis=1)
    x["cal_dm"] = x[["cal_ants_pd_approprié", "cal_ants_pd_réadapte"]].mean(axis=1)
    x["cal_tw"] = x[["cal_ants_eq_coordone", "cal_ants_eq_communique"]].mean(axis=1)
    x["cal_tm"] = x["cal_ants_gt_priorise"]
    x["game_sa"] = x[["game_ants_cs_changement_situatoin", "game_ants_cs_anticipe"]].mean(axis=1)
    x["game_dm"] = x[["game_ants_pd_approprié", "game_ants_pd_réadapte"]].mean(axis=1)
    x["game_tw"] = x[["game_ants_te_coordone", "game_ans_te_comunique"]].mean(axis=1)
    x["game_tm"] = x["game_ants_gt_priorise"]
    x["cal_ants"] = x[["cal_sa", "cal_dm", "cal_tw", "cal_tm"]].sum(axis=1)
    x["game_ants"] = x[["game_sa", "game_dm", "game_tw", "game_tm"]].sum(axis=1)
    for dmn in ["sa", "dm", "tw", "tm", "ants"]:
        x[f"delta_{dmn}"] = x[f"game_{dmn}"] - x[f"cal_{dmn}"]
    # task outcomes: calibration phase per participant (sub-team result), experimental phase per session
    x["cal_steps"] = x[["cal_etape_1min", "cal_etape_2min", "cal_etape_3min", "cal_etape_4min", "cal_etape_5min"]].sum(axis=1)
    x["cal_victory"] = x["cal_succes"].astype(int)
    x["game_steps"] = x["game_sucess"].astype(str).str[0].astype(int)
    x["game_victory"] = (x["game_steps"] == 5).astype(int)
    x["alarm"] = x["game_ants_gt_utilisation_alarme"].astype(int)
    # stress, peritraumatic distress, self-assessment, satisfaction, characteristics
    x["cal_stress"] = x["cal_eva_stress"]; x["game_stress"] = x["game_eva_stress"]; x["delta_stress"] = x["game_stress"] - x["cal_stress"]
    pdi_items = ["0_a_4_impression_que_ma_vie_etait_entrain_de_changer", "0_a_4_sentiment_d_impuissance", "0_a_4_j_ai_ete_terrifie",
                 "0_a_4_palpitations", "0_a_4_sueurs_tremblements", "0_a_4_sensation_d_etouffer", "0_a_4_dissociation",
                 "0_a_4_honte_ou_culpabilité_par_rapport_au_jeu", "0_a_4_difficulte_de_controle_des_emotions",
                 "0_a_4_precoccupe_par_l_idee_de_perdre_le_controle_de_la_situation", "0_a_4_incapable_de_me_concentrer"]
    for i, it in enumerate(pdi_items, 1):
        x[f"pdi_{i:02d}"] = x[it]
    x["pdi"] = x[pdi_items].sum(axis=1, min_count=len(pdi_items))
    x["self_sa"] = x["0_a4_Conscience_de_la_situation"]; x["self_dm"] = x["0_a_4_Prise_de_decision"]
    x["self_tw"] = x["0_a_4_Travail_en_equipe"]; x["self_tm"] = x["0_a_4_Gestion_de_la_tache"]
    x["satisfied"] = x["Avez_vous_apprecie_cette_experience_pedagogique"].map({"Oui": 1, "Non": 0})
    x["level"] = x["niveau_de_deuxieme_cycle"].map({"FASM1": "DFASM1", "FASM2": "DFASM2", "FASM3": "DFASM3"})
    for c, n in [("deja_realiser_un_escape_game", "prior_escape"), ("deja_realiser_simulation_medicale", "prior_sim"),
                 ("vecu_situation_de_stress_medicale", "prior_med_stress"), ("vecu_situation_de_stress_non_medicale", "prior_nonmed_stress")]:
        x[n] = x[c].map({"Oui": 1, "Non": 0})
    x["questionnaire_missing"] = x["Date"].isna() | x["Date"].astype(str).str.contains("manquante")
    # de-identification: sessions in chronological order -> S01..S16, dates -> D01..; the allocation stage is
    # derived from the date (the 2 sessions of the last study day were allocated in the completion stage)
    order = x.groupby("session_id")["session_date"].first().sort_values()
    x["team"] = x["session_id"].map({s: f"S{i+1:02d}" for i, s in enumerate(order.index)})
    dates = sorted(x["session_date"].dt.date.unique())
    x["date"] = x["session_date"].dt.date.map({dt: f"D{i+1:02d}" for i, dt in enumerate(dates)})
    x["allocation_stage"] = np.where(x["session_date"].dt.date == dates[-1], "completion", "unrestricted")
    x = x.sort_values(["team", "session_date"]).reset_index(drop=True)
    x["pid"] = np.arange(len(x)) + 1
    keep = ["pid", "team", "arm", "date", "allocation_stage", "level", "prior_escape", "prior_sim", "prior_med_stress", "prior_nonmed_stress",
            "cal_sa", "cal_dm", "cal_tw", "cal_tm", "cal_ants", "game_sa", "game_dm", "game_tw", "game_tm", "game_ants",
            "delta_sa", "delta_dm", "delta_tw", "delta_tm", "delta_ants", "cal_steps", "cal_victory", "game_steps", "game_victory", "alarm",
            "cal_stress", "game_stress", "delta_stress"] + [f"pdi_{i:02d}" for i in range(1, 12)] + \
           ["pdi", "self_sa", "self_dm", "self_tw", "self_tm", "satisfied", "questionnaire_missing"]
    x[["session_id", "team"]].drop_duplicates().to_csv(f"{DATA}/session_id_map.csv", index=False)  # kept locally only (git-ignored)
    return x[keep].copy()

if os.path.exists(RAW):
    d = build_analysis(RAW); d.to_csv(ANALYSIS, index=False); print("analysis.csv built from", RAW)
else:
    assert os.path.exists(ANALYSIS), f"Neither {RAW} nor {ANALYSIS} found"
    d = pd.read_csv(ANALYSIS); print("loaded", ANALYSIS)
d["questionnaire_missing"] = d["questionnaire_missing"].astype(bool)
# consistency checks: one arm per session; session-level outcomes constant within session
chk = d.groupby("team").agg(n_arms=("arm", "nunique"), n_steps=("game_steps", "nunique"), n_alarm=("alarm", "nunique"))
assert (chk == 1).all().all(), chk[(chk != 1).any(axis=1)]
assert d["cal_ants"].between(0, 16).all() and d["game_ants"].between(0, 16).all()
print(d.shape, d.arm.value_counts().to_dict(), "| sessions:", d.team.nunique(), "| questionnaires missing:", int(d.questionnaire_missing.sum()))

analysis.csv built from data/df.xlsx
(99, 51) {'Active': 33, 'Control': 33, 'Passive': 33} | sessions: 16 | questionnaires missing: 5


## 2. Cluster structure, ICC and design effect

One row per session. The ICC comes from null random-intercept models (`lmm_cluster.R`); the design effect is 1 + (m − 1) × ICC with m the mean session size, and the effective sample size n / design effect. Because the ICC of an outcome measured after the intervention also contains the intervention effect, the ICC adjusted for the allocated arm is reported alongside (Supplementary Table 3).

In [3]:
t = d.groupby(["team", "arm"]).agg(n=("pid", "size"), date=("date", "first"), allocation_stage=("allocation_stage", "first"),
                                   cal=("cal_ants", "mean"), game=("game_ants", "mean"), delta=("delta_ants", "mean"),
                                   cal_steps_mean=("cal_steps", "mean"), cal_steps_best=("cal_steps", "max"), cal_victory=("cal_victory", "max"),
                                   steps=("game_steps", "max"), victory=("game_victory", "max"), alarm=("alarm", "max"),
                                   stress=("game_stress", "mean")).reset_index()
t["arm"] = pd.Categorical(t["arm"], ARMS); t = t.sort_values(["arm", "team"]).reset_index(drop=True)
t.to_csv(f"{OUT}/session_level.csv", index=False)
print(t.round(2).to_string())
sessions_per_arm = t.groupby("arm", observed=True).agg(sessions=("team", "size"), participants=("n", "sum"), size_median=("n", "median"),
                                                      size_q1=("n", lambda s: s.quantile(.25)), size_q3=("n", lambda s: s.quantile(.75)),
                                                      size_min=("n", "min"), size_max=("n", "max"))
print(sessions_per_arm)

   team      arm  n date allocation_stage    cal   game  delta  cal_steps_mean  cal_steps_best  cal_victory  steps  victory  alarm  stress
0   S02  Control  6  D01     unrestricted   9.08   6.58  -2.50            3.00               5            1      2        0      0    2.50
1   S05  Control  8  D03     unrestricted  10.75   4.00  -6.75            2.50               3            0      0        0      0    2.38
2   S09  Control  7  D05     unrestricted   9.93   9.00  -0.93            2.00               2            0      2        0      1    3.43
3   S12  Control  6  D06     unrestricted   9.75   6.17  -3.58            2.00               2            0      2        0      0    2.50
4   S14  Control  6  D07     unrestricted   8.67  10.67   2.00            3.00               5            1      3        0      0    2.33
5   S03  Passive  6  D02     unrestricted  10.83  10.83   0.00            4.00               5            1      2        0      1    2.83
6   S07  Passive  3  D04   

In [4]:
# Mixed models (R). Results are read back as CSV files in outputs/.
res = subprocess.run([rscript, "lmm_cluster.R", ANALYSIS, OUT], capture_output=True, text=True)
print(res.stdout[-300:]); assert res.returncode == 0, res.stderr[-2000:]
icc = pd.read_csv(f"{OUT}/lmm_icc.csv"); anov = pd.read_csv(f"{OUT}/lmm_anova.csv"); emm = pd.read_csv(f"{OUT}/lmm_emm.csv"); con = pd.read_csv(f"{OUT}/lmm_contrasts.csv")
mbar = t["n"].mean()
icc["design_effect"] = 1 + (mbar - 1) * icc["icc"]; icc["effective_n"] = len(d) / icc["design_effect"]
icc.insert(0, "label", icc["outcome"].map({"cal_ants": "ANTS total, calibration phase", "game_ants": "ANTS total, experimental phase",
                                           "delta_ants": "ANTS total, change (primary outcome)", "game_stress": "Stress NRS, experimental phase", "pdi": "PDI"}))
print(f"mean session size m = {mbar:.2f}"); print(icc.round(3).to_string(index=False))
icc.to_csv(f"{OUT}/suppTable3_icc_design_effect.csv", index=False)

Mixed models done: 99 participants, 16 sessions -> outputs 

mean session size m = 6.19
                               label     outcome   icc  icc_adjusted  design_effect  effective_n
       ANTS total, calibration phase    cal_ants 0.143         0.136          1.741       56.875
      ANTS total, experimental phase   game_ants 0.757         0.633          4.926       20.099
ANTS total, change (primary outcome)  delta_ants 0.578         0.411          3.999       24.756
      Stress NRS, experimental phase game_stress 0.052         0.030          1.268       78.094
                                 PDI         pdi 0.323         0.347          2.673       37.037


## 3. Mixed-model results (primary and secondary outcomes)

Linear mixed models with the arm as fixed effect and a random intercept for session (REML, Kenward-Roger degrees of freedom, Tukey-adjusted pairwise contrasts). Phase-2 scores are adjusted for the corresponding phase-1 score (ANCOVA); change scores, PDI and self-ratings are unadjusted. A sensitivity model adds a random intercept for the session date (site proxy).

In [5]:
def fmt_p(p):
    if p < .001: return "<.001"
    s = f"{p:.3f}" if p < .01 else f"{p:.2f}"
    return s.lstrip("0")
con["ci"] = con.apply(lambda r: f"{r.estimate:+.2f} ({r.lower:.2f} to {r.upper:.2f})", axis=1)
con["P"] = con["p_tukey"].map(fmt_p)
print(anov.round(4).to_string(index=False)); print()
print(emm[emm.outcome == "ANTS total (phase 2, adjusted for phase 1)"][["arm", "emmean", "SE"]].round(2).to_string(index=False)); print()
print(con[["outcome", "contrast", "ci", "df", "P", "p_raw"]].round(4).to_string(index=False))

                                       outcome       F  df1     df2      p
    ANTS total (phase 2, adjusted for phase 1)  6.5101    2 12.9362 0.0111
                     ANTS total (change score)  6.8423    2 12.8676 0.0095
       Situation awareness (phase 2, adjusted) 11.0865    2 12.7516 0.0016
           Decision making (phase 2, adjusted)  5.4606    2 12.9824 0.0190
              Team working (phase 2, adjusted)  7.5377    2 12.8822 0.0068
           Task management (phase 2, adjusted)  2.9919    2 12.9483 0.0855
            Situation awareness (change score)  8.4813    2 12.6512 0.0046
                Decision making (change score)  7.7916    2 12.8225 0.0061
                   Team working (change score)  7.4523    2 12.8200 0.0071
                Task management (change score)  3.3994    2 12.8771 0.0652
                Stress NRS (phase 2, adjusted)  1.6425    2 11.6311 0.2353
                     Stress NRS (change score)  0.3440    2 12.6109 0.7154
              Peritraumat

## 4. Session-level analyses (one observation per session)

Sensitivity analysis of the primary outcome with the session mean of the ANTS change (Kruskal-Wallis, Mann-Whitney U and exact permutation tests respecting the cluster randomisation), and the task outcomes, which are intrinsically session-level in the experimental phase (Fisher-Freeman-Halton exact test across arms, pairwise Fisher exact tests, Kruskal-Wallis and Mann-Whitney U). In the calibration phase each of the two sub-teams had its own result; sub-team membership was not recorded, so the calibration task is summarised as the mean over the participants of the session and as the result of the better sub-team.

In [6]:
def perm_test(a, b, n=50000):
    obs = a.mean() - b.mean(); pool = np.r_[a, b]; k = len(a); cnt = 0
    for _ in range(n):
        rng.shuffle(pool); cnt += abs(pool[:k].mean() - pool[k:].mean()) >= abs(obs) - 1e-9
    return obs, cnt / n

def fisher_exact_rxc(ct):
    # Fisher-Freeman-Halton exact test for an r x c table, via R's fisher.test
    ct = np.asarray(ct); s = ";".join(",".join(str(int(v)) for v in row) for row in ct)
    r = subprocess.run([rscript, "-e", f'm<-do.call(rbind,lapply(strsplit("{s}",";")[[1]],function(z)as.integer(strsplit(z,",")[[1]])));cat(fisher.test(m,workspace=2e7)$p.value)'],
                       capture_output=True, text=True)
    assert r.returncode == 0, r.stderr
    return float(r.stdout.strip())

rows = []
for var, lab in [("delta", "Session-mean ANTS change"), ("game", "Session-mean ANTS phase 2"), ("cal", "Session-mean ANTS phase 1"),
                 ("steps", "Steps completed (phase 2)"), ("cal_steps_mean", "Steps completed (phase 1, session mean)"),
                 ("cal_steps_best", "Steps completed (phase 1, better sub-team)"), ("stress", "Session-mean stress NRS phase 2")]:
    kw = stats.kruskal(*[t[var][t.arm == a] for a in ARMS])
    for a, b in PAIRS:
        xa, xb = t[var][t.arm == a].values.astype(float), t[var][t.arm == b].values.astype(float)
        mw = stats.mannwhitneyu(xa, xb); o, pp = perm_test(xa, xb)
        rows.append(dict(outcome=lab, contrast=f"{a} vs {b}", median_a=np.median(xa), median_b=np.median(xb), mean_diff=o, kw_p=kw.pvalue, mwu_p=mw.pvalue, perm_p=pp))
team_tests = pd.DataFrame(rows); print(team_tests.round(4).to_string(index=False)); team_tests.to_csv(f"{OUT}/session_level_tests.csv", index=False)

                                   outcome           contrast  median_a  median_b  mean_diff   kw_p  mwu_p  perm_p
                  Session-mean ANTS change Passive vs Control    0.5938   -2.5000     3.3906 0.0245 0.1255  0.0777
                  Session-mean ANTS change  Active vs Control    3.7143   -2.5000     5.9066 0.0245 0.0159  0.0166
                  Session-mean ANTS change  Active vs Passive    3.7143    0.5938     2.5161 0.0245 0.1255  0.0903
                 Session-mean ANTS phase 2 Passive vs Control    9.9167    6.5833     2.8139 0.0344 0.0996  0.0592
                 Session-mean ANTS phase 2  Active vs Control   12.5000    6.5833     4.8622 0.0344 0.0317  0.0230
                 Session-mean ANTS phase 2  Active vs Passive   12.5000    9.9167     2.0483 0.0344 0.1432  0.1129
                 Session-mean ANTS phase 1 Passive vs Control    9.3438    9.7500    -0.5767 0.3919 0.9271  0.5721
                 Session-mean ANTS phase 1  Active vs Control    8.8333    9.750

In [7]:
brows = []
for var, lab in [("victory", "Victory (all 5 steps), phase 2"), ("cal_victory", "Victory (better sub-team), phase 1"), ("alarm", "Alarm silenced, phase 2")]:
    tab = t.groupby("arm", observed=True)[var].agg(["sum", "size"])
    p3 = fisher_exact_rxc(np.c_[tab["sum"], tab["size"] - tab["sum"]])
    for a, b in PAIRS:
        ct = [[tab.loc[a, "sum"], tab.loc[a, "size"] - tab.loc[a, "sum"]], [tab.loc[b, "sum"], tab.loc[b, "size"] - tab.loc[b, "sum"]]]
        brows.append(dict(outcome=lab, contrast=f"{a} vs {b}", a=f"{tab.loc[a,'sum']}/{tab.loc[a,'size']}", b=f"{tab.loc[b,'sum']}/{tab.loc[b,'size']}", fisher_p=stats.fisher_exact(ct)[1], global_p=p3))
    oth = tab.loc[["Passive", "Control"]].sum()
    ct = [[tab.loc["Active", "sum"], tab.loc["Active", "size"] - tab.loc["Active", "sum"]], [oth["sum"], oth["size"] - oth["sum"]]]
    brows.append(dict(outcome=lab, contrast="Active vs Passive+Control", a=f"{tab.loc['Active','sum']}/{tab.loc['Active','size']}", b=f"{oth['sum']}/{oth['size']}", fisher_p=stats.fisher_exact(ct)[1], global_p=p3))
team_bin = pd.DataFrame(brows); print(team_bin.round(4).to_string(index=False)); team_bin.to_csv(f"{OUT}/session_level_binary.csv", index=False)

                           outcome                  contrast   a    b  fisher_p  global_p
    Victory (all 5 steps), phase 2        Passive vs Control 0/6  0/5    1.0000    0.0357
    Victory (all 5 steps), phase 2         Active vs Control 3/5  0/5    0.1667    0.0357
    Victory (all 5 steps), phase 2         Active vs Passive 3/5  0/6    0.0606    0.0357
    Victory (all 5 steps), phase 2 Active vs Passive+Control 3/5 0/11    0.0179    0.0357
Victory (better sub-team), phase 1        Passive vs Control 2/6  2/5    1.0000    0.4643
Victory (better sub-team), phase 1         Active vs Control 0/5  2/5    0.4444    0.4643
Victory (better sub-team), phase 1         Active vs Passive 0/5  2/6    0.4545    0.4643
Victory (better sub-team), phase 1 Active vs Passive+Control 0/5 4/11    0.2445    0.4643
           Alarm silenced, phase 2        Passive vs Control 3/6  1/5    0.5455    0.2230
           Alarm silenced, phase 2         Active vs Control 4/5  1/5    0.2063    0.2230
          

In [8]:
# Table 3: session-level outcomes
tb = team_bin.set_index(["outcome", "contrast"]); tt = team_tests.set_index(["outcome", "contrast"])
def med(s):
    s = s.dropna(); f = lambda v: f"{round(v, 2):g}"; return f"{f(s.median())} [{f(s.quantile(.25))}–{f(s.quantile(.75))}]"
CONTRASTS = ["Passive vs Control", "Active vs Control", "Active vs Passive"]
T3 = [["Outcome (unit: session)", *[f"{a} ({(t.arm==a).sum()} sessions)" for a in ARMS], *CONTRASTS]]
for o, v, lab in [("Victory (better sub-team), phase 1", "cal_victory", "Calibration phase: sessions in which at least one sub-team completed all 5 steps, n/N"),
                  ("Victory (all 5 steps), phase 2", "victory", "Experimental phase: victory (all 5 steps), n/N"),
                  ("Alarm silenced, phase 2", "alarm", "Experimental phase: alarm silenced, n/N")]:
    T3.append([lab, *[f"{int(t[v][t.arm==a].sum())}/{(t.arm==a).sum()}" for a in ARMS], *["P=" + fmt_p(tb.loc[(o, c), "fisher_p"]) for c in CONTRASTS]])
for o, v, lab in [("Steps completed (phase 1, session mean)", "cal_steps_mean", "Calibration phase: steps completed, mean per session, median [IQR]"),
                  ("Steps completed (phase 1, better sub-team)", "cal_steps_best", "Calibration phase: steps completed by the better sub-team, median [IQR]"),
                  ("Steps completed (phase 2)", "steps", "Experimental phase: steps completed, median [IQR]"),
                  ("Session-mean ANTS change", "delta", "Session-mean ANTS change, median [IQR]")]:
    T3.append([lab, *[med(t[v][t.arm == a]) for a in ARMS], *["P=" + fmt_p(tt.loc[(o, c), "mwu_p"]) for c in CONTRASTS]])
table3 = pd.DataFrame(T3[1:], columns=T3[0]); table3.to_csv(f"{OUT}/table3_session_level.csv", index=False); print(table3.to_string(index=False))
print("\nGlobal tests across arms:",
      "victory phase 2 P=" + fmt_p(tb.loc[("Victory (all 5 steps), phase 2", "Active vs Control"), "global_p"]) + " (Fisher-Freeman-Halton);",
      "alarm P=" + fmt_p(tb.loc[("Alarm silenced, phase 2", "Active vs Control"), "global_p"]) + ";",
      "steps phase 2 P=" + fmt_p(tt.loc[("Steps completed (phase 2)", "Active vs Control"), "kw_p"]) + " (Kruskal-Wallis);",
      "session-mean ANTS change P=" + fmt_p(tt.loc[("Session-mean ANTS change", "Active vs Control"), "kw_p"]) + " (Kruskal-Wallis)")

                                                              Outcome (unit: session) Control (5 sessions) Passive (6 sessions) Active (5 sessions) Passive vs Control Active vs Control Active vs Passive
Calibration phase: sessions in which at least one sub-team completed all 5 steps, n/N                  2/5                  2/6                 0/5             P=1.00             P=.44             P=.45
                                       Experimental phase: victory (all 5 steps), n/N                  0/5                  0/6                 3/5             P=1.00             P=.17             P=.06
                                              Experimental phase: alarm silenced, n/N                  1/5                  3/6                 4/5              P=.55             P=.21             P=.55
                   Calibration phase: steps completed, mean per session, median [IQR]            2.5 [2–3]        2.12 [2–3.56]           2 [1.5–3]             P=1.00             P=.67    

## 5. Sensitivity analysis: sessions allocated in the unrestricted stage only (Supplementary Table 6)

Allocation was unrestricted until an arm reached its per-arm target (first 14 sessions) and then constrained to complete the remaining arms (last 2 sessions, on the final study day). The primary and session-level analyses are repeated after excluding the 2 sessions allocated in the completion stage.

In [9]:
ds = d[d["allocation_stage"] == "unrestricted"].copy()
SENS = f"{DATA}/analysis_sens.csv"; ds.to_csv(SENS, index=False)
res = subprocess.run([rscript, "lmm_cluster.R", SENS, OUT, "sens_"], capture_output=True, text=True); assert res.returncode == 0, res.stderr[-2000:]
con_s = pd.read_csv(f"{OUT}/sens_lmm_contrasts.csv"); anov_s = pd.read_csv(f"{OUT}/sens_lmm_anova.csv"); icc_s = pd.read_csv(f"{OUT}/sens_lmm_icc.csv")
ts = t[t["allocation_stage"] == "unrestricted"].copy()
rows_s = []
for out_label, key in [("ANTS total (phase 2, adjusted for phase 1)", "ANTS total (phase 2, adjusted for phase 1)"), ("ANTS total change", "ANTS total (change score)")]:
    for c in CONTRASTS:
        r_ = con_s[(con_s.outcome == key) & (con_s.contrast == c)].iloc[0]
        rows_s.append(dict(Analysis=out_label, Contrast=c, Estimate=f"{r_.estimate:+.2f} ({r_.lower:.2f} to {r_.upper:.2f})", P=fmt_p(r_.p_tukey)))
for var, lab in [("delta", "Session-mean ANTS change"), ("steps", "Steps completed (phase 2)")]:
    for a, b in PAIRS:
        p = stats.mannwhitneyu(ts[var][ts.arm == a], ts[var][ts.arm == b]).pvalue
        rows_s.append(dict(Analysis=f"{lab} (session level)", Contrast=f"{a} vs {b}",
                           Estimate=f"{round(np.median(ts[var][ts.arm == a]), 2):g} vs {round(np.median(ts[var][ts.arm == b]), 2):g}", P=fmt_p(p)))
tabv = ts.groupby("arm", observed=True)["victory"].agg(["sum", "size"])
ct = [[tabv.loc["Active", "sum"], tabv.loc["Active", "size"] - tabv.loc["Active", "sum"]],
      [tabv.loc[["Passive", "Control"], "sum"].sum(), (tabv.loc[["Passive", "Control"], "size"] - tabv.loc[["Passive", "Control"], "sum"]).sum()]]
rows_s.append(dict(Analysis="Victory, phase 2 (session level)", Contrast="Active vs Passive+Control",
                   Estimate=f"{int(tabv.loc['Active','sum'])}/{int(tabv.loc['Active','size'])} vs {int(tabv.loc[['Passive','Control'],'sum'].sum())}/{int(tabv.loc[['Passive','Control'],'size'].sum())} sessions",
                   P=fmt_p(stats.fisher_exact(ct)[1])))
supp_sens = pd.DataFrame(rows_s)
print(f"Sensitivity set: {len(ds)} students in {ts.shape[0]} sessions", ds.arm.value_counts().to_dict())
print("omnibus P (phase-2 model):", fmt_p(anov_s.set_index("outcome").loc["ANTS total (phase 2, adjusted for phase 1)", "p"]), "| ICC (change):", round(icc_s.set_index("outcome").loc["delta_ants", "icc"], 2))
print(supp_sens.to_string(index=False)); supp_sens.to_csv(f"{OUT}/suppTable6_sensitivity_allocation.csv", index=False)

Sensitivity set: 88 students in 14 sessions {'Control': 33, 'Passive': 29, 'Active': 26}
omnibus P (phase-2 model): .03 | ICC (change): 0.59
                                  Analysis                  Contrast              Estimate    P
ANTS total (phase 2, adjusted for phase 1)        Passive vs Control +2.38 (-1.37 to 6.12)  .24
ANTS total (phase 2, adjusted for phase 1)         Active vs Control  +4.66 (0.69 to 8.63)  .02
ANTS total (phase 2, adjusted for phase 1)         Active vs Passive +2.28 (-1.69 to 6.26)  .31
                         ANTS total change        Passive vs Control +3.33 (-1.38 to 8.04)  .18
                         ANTS total change         Active vs Control +5.84 (0.85 to 10.83)  .02
                         ANTS total change         Active vs Passive +2.51 (-2.50 to 7.52)  .40
  Session-mean ANTS change (session level)        Passive vs Control          0.19 vs -2.5  .15
  Session-mean ANTS change (session level)         Active vs Control          3.62 vs -2.5 

## 6. Descriptive tables

**Table 1** (characteristics and calibration-phase measures; Kruskal-Wallis for continuous variables and Fisher-Freeman-Halton exact test for categorical variables at the participant level, Kruskal-Wallis at the session level for session size and calibration-task steps), **Table 2** (experimental-phase outcomes with the mixed-model contrasts), **Supplementary Table 4** (PDI items, exploratory participant-level Kruskal-Wallis), **Supplementary Table 5** (initially planned participant-level analysis, Mann-Whitney U with Bonferroni correction) and individual-level counts reported in the text.

In [10]:
def npct(s, val=1, respondents=False):
    n = int(s.notna().sum()) if respondents else len(s); k = int((s == val).sum()); return f"{k} ({100*k/n:.1f})"
def kw_p(var): return fmt_p(stats.kruskal(*[d[var][d.arm == a].dropna() for a in ARMS]).pvalue)
def fisher_p(var): return fmt_p(fisher_exact_rxc(pd.crosstab(d[var], d["arm"])[ARMS].values))
def row(label, f, p=""): return [label, f(d), *[f(d[d.arm == a]) for a in ARMS], p]
T1 = [["Characteristic", f"Overall (n={len(d)})", *[f"{a} (n={(d.arm==a).sum()})" for a in ARMS], "P value"]]
T1 += [["Sessions (teams), n", str(t.shape[0]), *[str((t.arm == a).sum()) for a in ARMS], "—"]]
T1 += [["Session size, median [IQR]", med(t.n), *[med(t.n[t.arm == a]) for a in ARMS], fmt_p(stats.kruskal(*[t.n[t.arm == a] for a in ARMS]).pvalue)]]
T1 += [["Year of study, n (%)", "", "", "", "", fisher_p("level")]]
for lv, lab in [("DFASM1", "DFASM1 (4th year)"), ("DFASM2", "DFASM2 (5th year)"), ("DFASM3", "DFASM3 (6th year)")]:
    T1.append(row(f"   {lab}", lambda s, lv=lv: npct(s["level"], lv)))
for var, lab in [("prior_escape", "Prior escape-game participation, n (%)"), ("prior_sim", "Prior medical simulation, n (%)"),
                 ("prior_med_stress", "Prior stressful medical situation, n (%)"), ("prior_nonmed_stress", "Prior stressful non-medical situation, n (%)")]:
    T1.append(row(lab, lambda s, v=var: npct(s[v]), fisher_p(var)))
for var, lab in [("cal_sa", "ANTS situational awareness"), ("cal_dm", "ANTS decision-making"), ("cal_tw", "ANTS team working"), ("cal_tm", "ANTS task management"), ("cal_ants", "ANTS total"), ("cal_stress", "Stress numeric rating scale")]:
    T1.append(row(lab + ", median [IQR]", lambda s, v=var: med(s[v]), kw_p(var)))
T1.append(["Steps completed during the calibration task, mean per session, median [IQR]", med(t.cal_steps_mean), *[med(t.cal_steps_mean[t.arm == a]) for a in ARMS],
           fmt_p(stats.kruskal(*[t.cal_steps_mean[t.arm == a] for a in ARMS]).pvalue)])
table1 = pd.DataFrame(T1[1:], columns=T1[0]); table1.to_csv(f"{OUT}/table1.csv", index=False); print(table1.to_string(index=False))

                                                             Characteristic Overall (n=99) Control (n=33) Passive (n=33) Active (n=33) P value
                                                        Sessions (teams), n             16              5              6             5       —
                                                 Session size, median [IQR]  6 [5.75–7.25]        6 [6–7]      6 [4.5–6]       7 [5–8]     .47
                                                       Year of study, n (%)                                                                .14
                                                          DFASM1 (4th year)      42 (42.4)      13 (39.4)      18 (54.5)     11 (33.3)        
                                                          DFASM2 (5th year)      39 (39.4)      16 (48.5)      12 (36.4)     11 (33.3)        
                                                          DFASM3 (6th year)      13 (13.1)        3 (9.1)        2 (6.1)      8 (24.2)        

In [11]:
def con_row(outcome):
    c = con[con.outcome == outcome].set_index("contrast")
    return {k: f"{c.loc[k,'estimate']:+.2f} ({c.loc[k,'lower']:.2f} to {c.loc[k,'upper']:.2f}); P={c.loc[k,'P']}" for k in CONTRASTS}
T2 = [["Outcome", *[f"{a} (n={(d.arm==a).sum()})" for a in ARMS], *CONTRASTS]]
def t2row(label, var, outcome):
    cr = con_row(outcome); T2.append([label, *[med(d[var][d.arm == a]) for a in ARMS], *[cr[c] for c in CONTRASTS]])
T2.append(["Experimental phase, median [IQR]", "", "", "", "", "", ""])
t2row("   ANTS situational awareness", "game_sa", "Situation awareness (phase 2, adjusted)")
t2row("   ANTS decision-making", "game_dm", "Decision making (phase 2, adjusted)")
t2row("   ANTS team working", "game_tw", "Team working (phase 2, adjusted)")
t2row("   ANTS task management", "game_tm", "Task management (phase 2, adjusted)")
t2row("   ANTS total", "game_ants", "ANTS total (phase 2, adjusted for phase 1)")
t2row("   Stress numeric rating scale", "game_stress", "Stress NRS (phase 2, adjusted)")
T2.append(["Change from calibration phase, median [IQR]", "", "", "", "", "", ""])
t2row("   ANTS total change (primary outcome)", "delta_ants", "ANTS total (change score)")
t2row("   Stress change", "delta_stress", "Stress NRS (change score)")
T2.append(["Self-assessment after the experimental phase, median [IQR]", "", "", "", "", "", ""])
t2row("   Situational awareness", "self_sa", "Self-rated situation awareness")
t2row("   Decision-making", "self_dm", "Self-rated decision making")
t2row("   Team working", "self_tw", "Self-rated team working")
t2row("   Task management", "self_tm", "Self-rated task management")
t2row("   Peritraumatic Distress Inventory", "pdi", "Peritraumatic Distress Inventory")
table2 = pd.DataFrame(T2[1:], columns=T2[0]); table2.to_csv(f"{OUT}/table2.csv", index=False); print(table2.to_string(index=False))
print("\nSelf-reported outcomes available:", {a: int(d.pdi[d.arm == a].notna().sum()) for a in ARMS})

                                                   Outcome Control (n=33) Passive (n=33) Active (n=33)            Passive vs Control             Active vs Control            Active vs Passive
                          Experimental phase, median [IQR]                                                                                                                                     
                                ANTS situational awareness        2 [1–2]    2.5 [2–2.5]   3 [2.5–3.5]  +0.66 (-0.01 to 1.32); P=.05  +1.23 (0.53 to 1.92); P=.001 +0.57 (-0.10 to 1.24); P=.10
                                      ANTS decision-making      2 [1–2.5]      2 [2–2.5]   3 [2.5–3.5]  +0.52 (-0.42 to 1.45); P=.34   +1.21 (0.24 to 2.19); P=.02 +0.70 (-0.24 to 1.64); P=.16
                                         ANTS team working    2 [1.5–2.5]      3 [2.5–3]   3.5 [2.5–4]   +1.04 (0.13 to 1.95); P=.02  +1.30 (0.35 to 2.24); P=.008 +0.26 (-0.65 to 1.17); P=.74
                                      AN

In [12]:
# Supplementary Table 4 (PDI items), individual-level counts, satisfaction, Supplementary Table 5 (initially planned analysis)
pdi_lab = ["Life was about to change", "Helplessness", "Terrified", "Palpitations", "Sweating/trembling", "Choking sensation", "Dissociation",
           "Shame or guilt about the game", "Difficulty controlling emotions", "Worried about losing control of the situation", "Unable to concentrate"]
pdi_cols = [f"pdi_{i:02d}" for i in range(1, 12)]
S = [["PDI item (0–4)", f"Overall (n={int(d['pdi'].notna().sum())})", *ARMS, "Kruskal-Wallis P"]]
for it, lab in zip(pdi_cols, pdi_lab):
    S.append([lab, med(d[it]), *[med(d[it][d.arm == a]) for a in ARMS], kw_p(it)])
S.append(["Total", med(d["pdi"]), *[med(d["pdi"][d.arm == a]) for a in ARMS], kw_p("pdi")])
supp_pdi = pd.DataFrame(S[1:], columns=S[0]); supp_pdi.to_csv(f"{OUT}/suppTable4_pdi_items.csv", index=False); print(supp_pdi.to_string(index=False)); print()
ind = [["Individual-level counts (for information only; not the unit of analysis)", *ARMS]]
for v, lab in [("game_victory", "Participants in a victorious session, phase 2, n (%)"), ("cal_victory", "Participants in a victorious sub-team, phase 1, n (%)"),
               ("alarm", "Participants in a session that silenced the alarm, n (%)"), ("satisfied", "Satisfied with the session, n (%) of respondents")]:
    ind.append([lab, *[npct(d[v][d.arm == a], respondents=(v == "satisfied")) for a in ARMS]])
ind.append(["Questionnaire not returned, n", *[str(int(d["questionnaire_missing"][d.arm == a].sum())) for a in ARMS]])
supp_ind = pd.DataFrame(ind[1:], columns=ind[0]); supp_ind.to_csv(f"{OUT}/supp_individual_level_counts.csv", index=False); print(supp_ind.to_string(index=False))
sat = d["satisfied"].dropna(); print(f"\nOverall satisfaction: {int(sat.sum())}/{len(sat)} ({100*sat.mean():.1f}%)\n")
naive = []
for v, lab in [("delta_ants", "ANTS total change"), ("game_ants", "ANTS total phase 2"), ("game_steps", "Steps completed")]:
    for a, b in PAIRS:
        p = stats.mannwhitneyu(d[v][d.arm == a], d[v][d.arm == b]).pvalue
        naive.append(dict(outcome=lab, contrast=f"{a} vs {b}", p_raw=p, p_bonferroni=min(1, 3 * p)))
supp_naive = pd.DataFrame(naive); supp_naive.to_csv(f"{OUT}/suppTable5_participant_level_analysis.csv", index=False); print(supp_naive.round(4).to_string(index=False))
with pd.ExcelWriter(f"{OUT}/all_tables.xlsx") as w:
    for name, df_ in [("Table1", table1), ("Table2", table2), ("Table3_session", table3), ("SuppT3_ICC", icc), ("SuppT4_PDI", supp_pdi), ("SuppT5_naive", supp_naive),
                      ("SuppT6_sensitivity", supp_sens), ("Individual_counts", supp_ind), ("LMM_contrasts", con), ("LMM_anova", anov), ("LMM_emm", emm),
                      ("Session_tests", team_tests), ("Session_binary", team_bin), ("Session_level", t)]:
        df_.to_excel(w, sheet_name=name, index=False)

                               PDI item (0–4) Overall (n=94)    Control      Passive     Active Kruskal-Wallis P
                     Life was about to change        0 [0–0]    0 [0–0]      0 [0–0]    0 [0–0]              .38
                                 Helplessness        1 [0–2]    1 [0–2]      1 [1–3]    1 [0–2]              .02
                                    Terrified        0 [0–0]    0 [0–0]      0 [0–0]    0 [0–0]              .63
                                 Palpitations        0 [0–1] 0 [0–0.25]      0 [0–1] 0 [0–0.75]              .72
                           Sweating/trembling        0 [0–0]    0 [0–0]      0 [0–0]    0 [0–0]              .29
                            Choking sensation        0 [0–0]    0 [0–0]      0 [0–0]    0 [0–0]             1.00
                                 Dissociation        0 [0–0]    0 [0–0]      0 [0–0]    0 [0–0]              .67
                Shame or guilt about the game        0 [0–1]    0 [0–1]      0 [0–1]    0 [0–0] 

## 7. Figures

Figure 1 (ANTS change, participants and session means, mixed-model contrasts), Figure 2 (session-level task performance), Supplementary Figure 1 (CONSORT flow diagram, cluster extension) and Supplementary Figure 4 (alarm silencing).

In [13]:
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "svg.fonttype": "none"})
def star(p): return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "ns"
def bracket(ax, x1, x2, y, text, h=0.02):
    yr = ax.get_ylim(); dh = (yr[1] - yr[0]) * h
    ax.plot([x1, x1, x2, x2], [y, y + dh, y + dh, y], lw=1, c="k")
    ax.text((x1 + x2) / 2, y + dh, text, ha="center", va="bottom", fontsize=9, fontweight="bold")
def save(fig, name):
    fig.savefig(f"{OUT}/{name}.png", dpi=300, bbox_inches="tight"); fig.savefig(f"{OUT}/{name}.svg", bbox_inches="tight"); plt.close(fig)

# Figure 1
FIG1 = {"change": {"ants": "ANTS total (change score)", "sa": "Situation awareness (change score)", "dm": "Decision making (change score)",
                   "tw": "Team working (change score)", "tm": "Task management (change score)",
                   "note": "linear mixed models of the change score with a random intercept for session"},
        "ancova": {"ants": "ANTS total (phase 2, adjusted for phase 1)", "sa": "Situation awareness (phase 2, adjusted)", "dm": "Decision making (phase 2, adjusted)",
                   "tw": "Team working (phase 2, adjusted)", "tm": "Task management (phase 2, adjusted)",
                   "note": "linear mixed models of the phase-2 score adjusted for the phase-1 score, with a random intercept for session"}}[FIG1_CONTRASTS]
def box_team(ax, var, ylabel, outcome, title=None):
    data = [d[var][d.arm == a].values for a in ARMS]
    bp = ax.boxplot(data, positions=[0, 1, 2], widths=0.55, patch_artist=True, showfliers=False, medianprops=dict(color="k", lw=1.5), whiskerprops=dict(lw=1), capprops=dict(lw=1))
    for patch, a in zip(bp["boxes"], ARMS): patch.set_facecolor(COLORS[a]); patch.set_alpha(0.55); patch.set_edgecolor("k")
    tm = d.groupby("team")[var].mean()
    for i, a in enumerate(ARMS):
        tv = tm[t.team[t.arm == a]].values; jit = rng.uniform(-0.18, 0.18, len(tv))
        ax.scatter(i + jit, tv, s=38, facecolor="white", edgecolor="k", zorder=3, lw=1.2)
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels([LABELS[a] for a in ARMS]); ax.set_ylabel(ylabel)
    if title: ax.set_title(title, fontsize=10, fontweight="bold")
    ax.axhline(0, color="grey", lw=0.8, ls="--", zorder=0)
    c = con[con.outcome == outcome].set_index("contrast")
    top = max(np.max(v) for v in data); rngy = top - min(np.min(v) for v in data)
    ax.set_ylim(min(np.min(v) for v in data) - 0.1 * rngy, top + 0.5 * rngy)
    y0 = top + 0.06 * rngy; step = 0.13 * rngy
    bracket(ax, 0, 1, y0, star(c.loc["Passive vs Control", "p_tukey"]))
    bracket(ax, 0, 2, y0 + step, star(c.loc["Active vs Control", "p_tukey"]))
    bracket(ax, 1, 2, y0 + 2 * step, star(c.loc["Active vs Passive", "p_tukey"]))
fig = plt.figure(figsize=(9, 9)); gs = fig.add_gridspec(3, 2, height_ratios=[1.3, 1, 1], hspace=0.55, wspace=0.3)
box_team(fig.add_subplot(gs[0, :]), "delta_ants", "Δ ANTS total (phase 2 − phase 1)", FIG1["ants"], "Anaesthetists' Non-Technical Skills: change from calibration to experimental phase")
for k, (v, lab, key) in enumerate([("delta_sa", "Situational awareness", "sa"), ("delta_tw", "Team working", "tw"), ("delta_tm", "Task management", "tm"), ("delta_dm", "Decision-making", "dm")]):
    box_team(fig.add_subplot(gs[1 + k // 2, k % 2]), v, "Δ domain score", FIG1[key], lab)
fig.text(0.01, 0.005, f"Boxes: individual participants (median, IQR, whiskers to 1.5×IQR). Open circles: session (team) means. Brackets: Tukey-adjusted contrasts from {FIG1['note']} (Kenward-Roger df). ns P≥.05; * P<.05; ** P<.01; *** P<.001.", fontsize=7.5, wrap=True)
save(fig, "Figure1_ANTS")

In [14]:
# Figure 2: session-level task performance
fig, axes = plt.subplots(1, 3, figsize=(11, 4), gridspec_kw=dict(width_ratios=[1, 1, 1.15], wspace=0.45))
for ax, v, ttl, oc in [(axes[0], "cal_victory", "Victory — calibration phase", "Victory (better sub-team), phase 1"), (axes[1], "victory", "Victory — experimental phase", "Victory (all 5 steps), phase 2")]:
    for i, a in enumerate(ARMS):
        k, n = int(t[v][t.arm == a].sum()), int((t.arm == a).sum())
        ax.bar(i, 100 * k / n, color=COLORS[a], edgecolor="k", width=0.6); ax.text(i, 100 * k / n + 3, f"{k}/{n}", ha="center", fontsize=9, fontweight="bold")
    ax.set_ylim(0, 115); ax.set_ylabel("Sessions achieving victory (%)"); ax.set_xticks([0, 1, 2]); ax.set_xticklabels([LABELS[a] for a in ARMS]); ax.set_title(ttl, fontsize=10, fontweight="bold")
    ax.text(1, 108, f"Fisher-Freeman-Halton exact test: P={fmt_p(tb.loc[(oc, 'Active vs Control'), 'global_p'])}", ha="center", fontsize=8.5)
ax = axes[2]
for i, a in enumerate(ARMS):
    vals = t["steps"][t.arm == a].values; jit = np.linspace(-0.2, 0.2, len(vals))
    ax.scatter(i + jit, vals, s=60, facecolor=COLORS[a], edgecolor="k", zorder=3); ax.hlines(np.median(vals), i - 0.3, i + 0.3, color="k", lw=2)
ax.set_ylim(-0.3, 6.6); ax.set_yticks(range(6)); ax.set_ylabel("Steps completed by the session (0–5)"); ax.set_xticks([0, 1, 2]); ax.set_xticklabels([LABELS[a] for a in ARMS])
ax.set_title("Steps completed — experimental phase", fontsize=10, fontweight="bold")
for (a, b, y) in [("Passive", "Control", 5.35), ("Active", "Control", 5.75), ("Active", "Passive", 6.15)]:
    bracket(ax, ARMS.index(b), ARMS.index(a), y, star(tt.loc[("Steps completed (phase 2)", f"{a} vs {b}"), "mwu_p"]), h=0.015)
fig.text(0.01, -0.04, f"Unit of analysis: session (Control {(t.arm=='Control').sum()}, Passive {(t.arm=='Passive').sum()}, Active {(t.arm=='Active').sum()} sessions). Points: sessions; bars: median. Brackets: pairwise Mann-Whitney U tests between sessions. ns P≥.05; * P<.05; ** P<.01; *** P<.001.", fontsize=7.5)
save(fig, "Figure2_task_performance")

# Supplementary Figure 4: alarm silencing per session
fig, ax = plt.subplots(figsize=(4.2, 4))
for i, a in enumerate(ARMS):
    k, n = int(t["alarm"][t.arm == a].sum()), int((t.arm == a).sum())
    ax.bar(i, 100 * k / n, color=COLORS[a], edgecolor="k", width=0.6); ax.text(i, 100 * k / n + 3, f"{k}/{n}", ha="center", fontsize=9, fontweight="bold")
ax.set_ylim(0, 115); ax.set_ylabel("Sessions that silenced the alarm (%)"); ax.set_xticks([0, 1, 2]); ax.set_xticklabels([LABELS[a] for a in ARMS])
ax.text(1, 108, f"Fisher-Freeman-Halton exact test: P={fmt_p(tb.loc[('Alarm silenced, phase 2', 'Active vs Control'), 'global_p'])}", ha="center", fontsize=8.5)
save(fig, "SuppFigure4_alarm")

In [15]:
# Supplementary Figure 1: CONSORT flow diagram (cluster randomised extension)
n_sess = {a: int((t.arm == a).sum()) for a in ARMS}; n_part = {a: int((d.arm == a).sum()) for a in ARMS}
n_q = {a: int((d.arm == a).sum() - d["questionnaire_missing"][d.arm == a].sum()) for a in ARMS}
n_enrolled = len(d) + sum(EXCLUDED.values())
assert n_enrolled == N_APPROACHED - N_NOT_ENROLLED
fig, ax = plt.subplots(figsize=(11, 9.5)); ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off")
def box(xc, yc, w, h, text, fs=8.5, bold=False):
    ax.add_patch(FancyBboxPatch((xc - w / 2, yc - h / 2), w, h, boxstyle="round,pad=0.3,rounding_size=0.8", fc="white", ec="k", lw=1.1))
    ax.text(xc, yc, text, ha="center", va="center", fontsize=fs, fontweight="bold" if bold else "normal", linespacing=1.35)
def arrow(x1, y1, x2, y2): ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=12, lw=1.1, color="k"))
def side(y, text): ax.text(2, y, text, fontsize=8, va="center", fontweight="bold", rotation=90, ha="center")
side(90, "Enrolment"); side(62, "Allocation"); side(37, "Follow-up"); side(12, "Analysis")
box(50, 93, 52, 9, f"Sessions organised: {t.shape[0]} (5 hospitals, April–August 2025)\nStudents approached: {N_APPROACHED}\nNot enrolled: {N_NOT_ENROLLED} (target sample size reached before the last session)\nEnrolled and consented: {n_enrolled} students in {t.shape[0]} sessions", 8.5)
box(50, 80, 40, 6, f"Sessions randomised: {t.shape[0]}\nCalibration phase (all sessions): {n_enrolled} students", 8.5, True)
arrow(50, 88.5, 50, 83.2)
xs = {"Control": 18, "Passive": 50, "Active": 82}
for a, name in zip(ARMS, ["Control\n(no prior instruction)", "Passive learning\n(didactic lecture)", "Active learning\n(facilitated discussion)"]):
    arrow(50, 77, xs[a], 70.5); ex = EXCLUDED.get(a, 0)
    box(xs[a], 64, 30, 12, f"Allocated to {name}\nSessions: {n_sess[a]}\nStudents: {n_part[a] + ex}\nSession size: {int(t.n[t.arm==a].min())}–{int(t.n[t.arm==a].max())}", 8.2, True)
    arrow(xs[a], 59, xs[a], 47.5)
    lost = f"Did not complete the full procedure: {ex}" + ("\n(left before the end of the session)" if ex else "")
    box(xs[a], 41, 30, 11, f"Received allocated intervention: {n_sess[a]} sessions\nExperimental phase completed\n{lost}\nQuestionnaire not returned: {n_part[a] - n_q[a]}", 8)
    arrow(xs[a], 35.5, xs[a], 23.5)
    box(xs[a], 15, 30, 14, f"Analysed (primary outcome, ANTS)\nSessions: {n_sess[a]}\nStudents: {n_part[a]}\nSelf-reported outcomes: {n_q[a]} students\nSession-level outcomes: {n_sess[a]} sessions", 8.2, True)
ax.text(50, 2.5, "Unit of randomisation and intervention: session (team). Unit of ANTS assessment: individual participant. Analysis: mixed models with a random intercept for session; session-level outcomes analysed with the session as the unit.", ha="center", fontsize=7.5, style="italic")
save(fig, "SuppFigure1_CONSORT")
print("Figures written to", OUT); print(sorted(f for f in os.listdir(OUT) if f.endswith(".png")))

Figures written to outputs
['Figure1_ANTS.png', 'Figure2_task_performance.png', 'SuppFigure1_CONSORT.png', 'SuppFigure4_alarm.png']


## 8. Verification against the manuscript

Key numbers as reported in the manuscript (abstract, Results, Tables 1–3, Supplementary Table 3). The cell fails if the pipeline no longer reproduces them.

In [16]:
def contrast(outcome, c): return con[(con.outcome == outcome) & (con.contrast == c)].iloc[0]
checks = {
    # design
    "99 participants, 33 per arm": (len(d) == 99) and (d.arm.value_counts() == 33).all(),
    "16 sessions: 5 control, 6 passive, 5 active": dict(t.arm.value_counts()) == {"Control": 5, "Passive": 6, "Active": 5},
    "session size 3-9, median 6 [5.75-7.25]": (t.n.min(), t.n.max(), t.n.median(), t.n.quantile(.25), t.n.quantile(.75)) == (3, 9, 6, 5.75, 7.25),
    "94 questionnaires": int((~d.questionnaire_missing).sum()) == 94,
    # clustering
    "ICC phase 1 / phase 2 / change = 0.14 / 0.76 / 0.58": tuple(icc.set_index("outcome").loc[["cal_ants", "game_ants", "delta_ants"], "icc"].round(2)) == (0.14, 0.76, 0.58),
    "ICC adjusted phase 2 / change = 0.63 / 0.41": tuple(icc.set_index("outcome").loc[["game_ants", "delta_ants"], "icc_adjusted"].round(2)) == (0.63, 0.41),
    "design effect 4.0, effective n ~25": (round(icc.set_index("outcome").loc["delta_ants", "design_effect"], 1), round(icc.set_index("outcome").loc["delta_ants", "effective_n"])) == (4.0, 25),
    # primary outcome
    "omnibus F(2,12.9)=6.51, P=.01": (round(anov.iloc[0].F, 2), round(anov.iloc[0].df2, 1), fmt_p(anov.iloc[0].p)) == (6.51, 12.9, ".01"),
    "adjusted means 7.25 / 10.09 / 12.13": tuple(emm[emm.outcome == "ANTS total (phase 2, adjusted for phase 1)"].set_index("arm").loc[ARMS, "emmean"].round(2)) == (7.25, 10.09, 12.13),
    "active vs control +4.87 (1.28 to 8.47), P=.009": (lambda r: (round(r.estimate, 2), round(r.lower, 2), round(r.upper, 2), fmt_p(r.p_tukey)))(contrast("ANTS total (phase 2, adjusted for phase 1)", "Active vs Control")) == (4.87, 1.28, 8.47, ".009"),
    "passive vs control +2.83 (-0.61 to 6.28), P=.11": (lambda r: (round(r.estimate, 2), round(r.lower, 2), round(r.upper, 2), fmt_p(r.p_tukey)))(contrast("ANTS total (phase 2, adjusted for phase 1)", "Passive vs Control")) == (2.83, -0.61, 6.28, ".11"),
    "active vs passive +2.04 (-1.41 to 5.49), P=.30": (lambda r: (round(r.estimate, 2), round(r.lower, 2), round(r.upper, 2), fmt_p(r.p_tukey)))(contrast("ANTS total (phase 2, adjusted for phase 1)", "Active vs Passive")) == (2.04, -1.41, 5.49, ".30"),
    "change score active vs control +5.89 (1.64 to 10.14), P=.008": (lambda r: (round(r.estimate, 2), round(r.lower, 2), round(r.upper, 2), fmt_p(r.p_tukey)))(contrast("ANTS total (change score)", "Active vs Control")) == (5.89, 1.64, 10.14, ".008"),
    "date sensitivity active vs control +4.87 (0.75 to 9.00), P=.02": (lambda r: (round(r.estimate, 2), round(r.lower, 2), round(r.upper, 2), fmt_p(r.p_tukey)))(contrast("ANTS total (sensitivity: + date random effect)", "Active vs Control")) == (4.87, 0.75, 9.0, ".02"),
    "median change -3.5 / +1.0 / +3.0": tuple(d.groupby("arm").delta_ants.median().loc[ARMS]) == (-3.5, 1.0, 3.0),
    # domains
    "SA active vs control +1.23, P=.001": (round(contrast("Situation awareness (phase 2, adjusted)", "Active vs Control").estimate, 2), fmt_p(contrast("Situation awareness (phase 2, adjusted)", "Active vs Control").p_tukey)) == (1.23, ".001"),
    "TW passive vs control +1.04, P=.02": (round(contrast("Team working (phase 2, adjusted)", "Passive vs Control").estimate, 2), fmt_p(contrast("Team working (phase 2, adjusted)", "Passive vs Control").p_tukey)) == (1.04, ".02"),
    # session level
    "session-mean change KW P=.02, active vs control MWU P=.02, permutation P=.02": (fmt_p(tt.loc[("Session-mean ANTS change", "Active vs Control"), "kw_p"]), fmt_p(tt.loc[("Session-mean ANTS change", "Active vs Control"), "mwu_p"]), fmt_p(tt.loc[("Session-mean ANTS change", "Active vs Control"), "perm_p"])) == (".02", ".02", ".02"),
    "victory 3/5 vs 0/11, global P=.04, active vs others P=.02, vs control P=.17, vs passive P=.06": (tb.loc[("Victory (all 5 steps), phase 2", "Active vs Passive+Control"), "a"], tb.loc[("Victory (all 5 steps), phase 2", "Active vs Passive+Control"), "b"], fmt_p(tb.loc[("Victory (all 5 steps), phase 2", "Active vs Control"), "global_p"]), fmt_p(tb.loc[("Victory (all 5 steps), phase 2", "Active vs Passive+Control"), "fisher_p"]), fmt_p(tb.loc[("Victory (all 5 steps), phase 2", "Active vs Control"), "fisher_p"]), fmt_p(tb.loc[("Victory (all 5 steps), phase 2", "Active vs Passive"), "fisher_p"])) == ("3/5", "0/11", ".04", ".02", ".17", ".06"),
    "steps KW P=.01, active vs control P=.01, vs passive P=.02, passive vs control P=.62": tuple(fmt_p(tt.loc[("Steps completed (phase 2)", c), k]) for c, k in [("Active vs Control", "kw_p"), ("Active vs Control", "mwu_p"), ("Active vs Passive", "mwu_p"), ("Passive vs Control", "mwu_p")]) == (".01", ".01", ".02", ".62"),
    "alarm 4/5, 3/6, 1/5, P=.22": (tuple(int(t.alarm[t.arm == a].sum()) for a in ARMS), fmt_p(tb.loc[("Alarm silenced, phase 2", "Active vs Control"), "global_p"])) == ((1, 3, 4), ".22"),
    "calibration steps per session P=.64": table1.iloc[-1, -1] == ".64",
    # Table 1 and satisfaction
    "Table 1 Fisher P: year .14, escape .37, simulation .52, medical stress .17, non-medical stress .66": list(table1.set_index("Characteristic").loc[["Year of study, n (%)", "Prior escape-game participation, n (%)", "Prior medical simulation, n (%)", "Prior stressful medical situation, n (%)", "Prior stressful non-medical situation, n (%)"], "P value"]) == [".14", ".37", ".52", ".17", ".66"],
    "baseline decision-making P=.01, ANTS total P=.16": (table1.set_index("Characteristic").loc["ANTS decision-making, median [IQR]", "P value"], table1.set_index("Characteristic").loc["ANTS total, median [IQR]", "P value"]) == (".01", ".16"),
    "satisfaction 90/94 (95.7%)": (int(sat.sum()), len(sat)) == (90, 94),
    "sensitivity (unrestricted allocation): 88 students in 14 sessions": (len(ds), ts.shape[0]) == (88, 14),
}
failed = [k for k, v in checks.items() if not bool(v)]
for k, v in checks.items(): print(("OK  " if v else "FAIL"), k)
assert not failed, f"{len(failed)} check(s) failed: {failed}"
print(f"\nAll {len(checks)} checks passed.")

OK   99 participants, 33 per arm
OK   16 sessions: 5 control, 6 passive, 5 active
OK   session size 3-9, median 6 [5.75-7.25]
OK   94 questionnaires
OK   ICC phase 1 / phase 2 / change = 0.14 / 0.76 / 0.58
OK   ICC adjusted phase 2 / change = 0.63 / 0.41
OK   design effect 4.0, effective n ~25
OK   omnibus F(2,12.9)=6.51, P=.01
OK   adjusted means 7.25 / 10.09 / 12.13
OK   active vs control +4.87 (1.28 to 8.47), P=.009
OK   passive vs control +2.83 (-0.61 to 6.28), P=.11
OK   active vs passive +2.04 (-1.41 to 5.49), P=.30
OK   change score active vs control +5.89 (1.64 to 10.14), P=.008
OK   date sensitivity active vs control +4.87 (0.75 to 9.00), P=.02
OK   median change -3.5 / +1.0 / +3.0
OK   SA active vs control +1.23, P=.001
OK   TW passive vs control +1.04, P=.02
OK   session-mean change KW P=.02, active vs control MWU P=.02, permutation P=.02
OK   victory 3/5 vs 0/11, global P=.04, active vs others P=.02, vs control P=.17, vs passive P=.06
OK   steps KW P=.01, active vs control 